# 🚀 COUNTERFACTUAL FRAUD MODEL - GETTING STARTED GUIDE

This comprehensive guide introduces newcomers to the counterfactual fraud model project.
It covers the main concepts, configuration classes, and pipelines with practical examples.

The framework is designed for off-policy evaluation of fraud detection models, allowing you to:
- Generate synthetic fraud data with realistic characteristics
- Apply logging policies (e.g., block high-risk transactions)  
- Estimate counterfactual metrics using different models and strategies
- Compare model performance and retraining approaches

🎯 What you'll learn:
1. Configuration system and main classes
2. Data generation approaches (basic vs synthetic)
3. Pipeline types and their use cases
4. Practical examples you can run immediately
5. How to analyze and interpret results

In [1]:
import sys
from pathlib import Path

# Add the project root to Python path (for Jupyter notebooks)
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

In [30]:
# Import all the main components
from counterfactual_fraud_model import (
    # Enums for configuration
    ModelType, PropensityType,
    
    # Configuration classes
    DataGeneratorConfig, SyntheticDataConfig, ModelConfig, 
    LoggingPolicyConfig, CounterfactualEstimatorConfig, PipelineConfig,
    OffPolicyEvaluationConfig, SyntheticOffPolicyEvaluationConfig, 
    SyntheticRetrainingConfig, RetrainingConfig,
    
    # Main pipeline classes
    OffPolicyEvaluationPipeline,
    SyntheticOffPolicyEvaluationPipeline, 
    SyntheticRetrainingPipeline
)

# Import additional config classes not in main __init__.py
from counterfactual_fraud_model.config import RetrainingStrategy, RetrainingModelConfig

# Import individual component classes for examples
from counterfactual_fraud_model.generators import DataGenerator, SyntheticDataGenerator, LoggingPolicyGenerator
from counterfactual_fraud_model.estimators import CounterfactualEstimator
from counterfactual_fraud_model.core import ModelTrainer

## Configuration Classes
The project uses Pydantic models and Enums for type-safe configuration.
All configurations have sensible defaults and validation.

### Enums
They manage which models, strategies and propensities have been implemeted so far.
- Model types: ML capables to fit in the current pipelines
- Propensity Types: approaches to make the stochastic exploration of blocked transactions
- Retraining Strategies: Strategies to use logged data (a.k.a production data generated from a previous model which blocked transactions based on a policy) to retrain a new model

In [5]:
print("Available model types:")
for model_type in ModelType:
    print(f"  • {model_type.value}")
print("Available Propensity Types:")
for model_type in PropensityType:
    print(f"  • {model_type.value}")
print("Retraining Strategies:")
for model_type in RetrainingStrategy:
    print(f"  • {model_type.value}")

Available model types:
  • lightgbm
  • random_forest
  • logistic
Available Propensity Types:
  • uniform
Retraining Strategies:
  • filtering
  • weighting
  • fraud_injection


### Basic Configs

These are the most primitive configs from the package. They rule direct behaviors from concrete implementations

**Data Generator Config**

This config manages the beta, normal parameters distributions used to generate synthetic model scores and fraud occurrences distribution

In [10]:
DataGeneratorConfig(
    alpha=0.1,           # Beta distribution parameter
    beta_param=2.0,      # Beta distribution parameter  
    mean=-0.5,           # model bias in relation to true fraud occurrence
    sd=0.5,              # Normal model error std
    sample_size=10_000,  # Number of samples
    random_state=42      # For reproducibility
)

DataGeneratorConfig(alpha=0.1, beta_param=2.0, mean=-0.5, sd=0.5, sample_size=10000, random_state=42)

**Synthetic Data Generator Config**

This config manages the parameters from sklearn.datasets.make_classification along with other relevante to generate synthetic data to train classification models (and make OPE/OPL over them)

In [9]:
SyntheticDataConfig(
    n_samples=50_000,      # Total samples
    n_features=20,         # Total features
    n_informative=15,      # Informative features
    n_redundant=3,         # Redundant features
    weights=[0.985, 0.015], # Class imbalance (98.5% legitimate, 1.5% fraud)
    class_sep=1.2,         # How separated classes are
    random_state=42
)

SyntheticDataConfig(n_samples=50000, n_features=20, n_informative=15, n_redundant=3, n_repeated=0, n_clusters_per_class=2, weights=[0.985, 0.015], flip_y=0.01, class_sep=1.2, test_size=0.5, random_state=42)

**Model Config**

This config manages the model type selected, hyperparameters, and random state

In [12]:
ModelConfig(
    model_type=ModelType.LIGHTGBM,
    model_params={
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1
    },
    random_state=42
)

ModelConfig(model_type=<ModelType.LIGHTGBM: 'lightgbm'>, model_params={'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.1}, random_state=42)

**Model Config**

This config manages how the Logging Policy (i.e. what transactions will be blocked based on the model score, and how the blocked will be explored) wil be executed

In [13]:
LoggingPolicyConfig(
    cutoff=0.05,              # Block transactions with score > 0.05
    exploration_rate=0.1,     # Allow 10% of blocked transactions for exploration
    propensity_type=PropensityType.UNIFORM, # Propensity type: how the stochasticity exploration strategy is implemented
    random_state=42
)

LoggingPolicyConfig(cutoff=0.05, exploration_rate=0.1, propensity_type=<PropensityType.UNIFORM: 'uniform'>, random_state=42)

**Counterfactual Estimator Config**

This config manages how the counterfactual estimators will be calculated. Currently, the unique parameter is the number of boostrap repetitions (highly adviceble use +5k)

In [14]:
CounterfactualEstimatorConfig(
    n_bootstrap=5000,        # Number of bootstrap samples for confidence intervals
    random_state=42          # For reproducible results
)

CounterfactualEstimatorConfig(n_bootstrap=5000, random_state=42)

### Pipeline Configs

Pipeline are objects made to orchestrate the primitives implementations, and unlock run them multiple times easily. Pipelines configs, as expected, create configs from primitive configs to control pipeline objects

**Pipeline Config**

This minor config rule if the synthetic generated on a pipeline executiomn should be retained

In [15]:
PipelineConfig(include_data=True)

PipelineConfig(include_data=True)

**Off Policy Evaluation Config**

This config controls a basic OPE pipeline made from DataGenerator objects (i.e. model scores and fraud occurrences are modeled from pdf distributional parameters)

In [16]:
OffPolicyEvaluationConfig(
    data_generator=DataGeneratorConfig(sample_size=20_000, random_state=42),
    logging_policy=LoggingPolicyConfig(cutoff=0.05, exploration_rate=0.1),
    counterfactual_estimator=CounterfactualEstimatorConfig(n_bootstrap=1000),
    pipeline=PipelineConfig(include_data=False)
)

OffPolicyEvaluationConfig(data_generator=DataGeneratorConfig(alpha=0.1, beta_param=2.0, mean=-0.5, sd=0.5, sample_size=20000, random_state=42), logging_policy=LoggingPolicyConfig(cutoff=0.05, exploration_rate=0.1, propensity_type=<PropensityType.UNIFORM: 'uniform'>, random_state=None), counterfactual_estimator=CounterfactualEstimatorConfig(n_bootstrap=1000, random_state=None), pipeline=PipelineConfig(include_data=False))

**Synthetic Off Policy Evaluation Config**

Similar to the above, but it uses a SyntheticDataConfig to generate a full dataset, and a ModelConfig to train a model and generate the model scores from it.

In [18]:
synthetic_ope_config = SyntheticOffPolicyEvaluationConfig(
    synthetic_data=SyntheticDataConfig(n_samples=50_000, n_features=20),
    model=ModelConfig(model_type=ModelType.LIGHTGBM),
    logging_policy=LoggingPolicyConfig(cutoff=0.05, exploration_rate=0.05),
    counterfactual_estimator=CounterfactualEstimatorConfig(n_bootstrap=5000),
    pipeline=PipelineConfig(include_data=False)
)
synthetic_ope_config

SyntheticOffPolicyEvaluationConfig(synthetic_data=SyntheticDataConfig(n_samples=50000, n_features=20, n_informative=15, n_redundant=5, n_repeated=0, n_clusters_per_class=2, weights=[0.985, 0.015], flip_y=0.01, class_sep=1.0, test_size=0.5, random_state=None), model=ModelConfig(model_type=<ModelType.LIGHTGBM: 'lightgbm'>, model_params={}, random_state=None), logging_policy=LoggingPolicyConfig(cutoff=0.05, exploration_rate=0.05, propensity_type=<PropensityType.UNIFORM: 'uniform'>, random_state=None), counterfactual_estimator=CounterfactualEstimatorConfig(n_bootstrap=5000, random_state=None), pipeline=PipelineConfig(include_data=False))

**Synthetic Off Policy Evaluation Config**

Higher Order Config. It orchestrates a SyntheticOffPolicyEvaluationConfig to generate a logging policy, and, additionally, a retraning strategy to be executed on the logged data generated by the initial model.

In [19]:
SyntheticRetrainingConfig(
        base_config=synthetic_ope_config,
        retraining=RetrainingConfig(
            retrain_test_size=0.3,
            retrain_model=RetrainingModelConfig(
                base_model=ModelConfig(model_type=ModelType.LIGHTGBM),
                strategy=RetrainingStrategy.FILTERING,
                classification_threshold=0.1
            )
        )
    )

SyntheticRetrainingConfig(base_config=SyntheticOffPolicyEvaluationConfig(synthetic_data=SyntheticDataConfig(n_samples=50000, n_features=20, n_informative=15, n_redundant=5, n_repeated=0, n_clusters_per_class=2, weights=[0.985, 0.015], flip_y=0.01, class_sep=1.0, test_size=0.5, random_state=None), model=ModelConfig(model_type=<ModelType.LIGHTGBM: 'lightgbm'>, model_params={}, random_state=None), logging_policy=LoggingPolicyConfig(cutoff=0.05, exploration_rate=0.05, propensity_type=<PropensityType.UNIFORM: 'uniform'>, random_state=None), counterfactual_estimator=CounterfactualEstimatorConfig(n_bootstrap=5000, random_state=None), pipeline=PipelineConfig(include_data=False)), retraining=RetrainingConfig(retrain_test_size=0.3, retrain_model=RetrainingModelConfig(base_model=ModelConfig(model_type=<ModelType.LIGHTGBM: 'lightgbm'>, model_params={}, random_state=None), strategy=<RetrainingStrategy.FILTERING: 'filtering'>, strategy_params={}, classification_threshold=0.1)))

## Concrete Classes
The pipelines use concrete implementation classes that handle specific tasks. Understanding these classes helps you customize behavior and debug issues.

**DataGenerator**

Generates synthetic model score and fraud ocurrences based on distributional parameters

In [31]:
data_gen = DataGenerator(DataGeneratorConfig(
    alpha=0.1,
    beta_param=2.0, 
    sample_size=1000,
    random_state=42
))
sample_data = data_gen.generate_data()

print("DataGenerator creates basic fraud datasets:")
print(f"  • Uses beta distribution (alpha={data_gen.config.alpha}, beta={data_gen.config.beta_param})")
print(f"  • Adds normal noise (mean={data_gen.config.mean}, sd={data_gen.config.sd})")
print(f"  • Generated {len(sample_data)} samples")
print(f"  • Fraud rate: {sample_data['is_fraud'].mean():.3f}")
print(f"  • Score range: [{sample_data['model_scores'].min():.3f}, {sample_data['model_scores'].max():.3f}]")

DataGenerator creates basic fraud datasets:
  • Uses beta distribution (alpha=0.1, beta=2.0)
  • Adds normal noise (mean=-0.5, sd=0.5)
  • Generated 1000 samples
  • Fraud rate: 0.056
  • Score range: [0.000, 0.891]


**DataGenerator**

Generates synthetic dataset and model scores based on SyntheticDataConfig and ModelConfig

In [35]:
synthetic_gen = SyntheticDataGenerator(
    SyntheticDataConfig(
        n_samples=2000,
        n_features=10,
        n_informative=6,
        n_redundant=2,
        random_state=42
    ),
    ModelConfig(model_type=ModelType.LIGHTGBM, random_state=42)
)
synthetic_data = synthetic_gen.generate_data()

print("SyntheticDataGenerator creates ML-ready datasets:")
print(f"  • Uses sklearn.make_classification for realistic features")
print(f"  • Trains {synthetic_gen.model_config.model_type.value} model automatically")
print(f"  • Generated {len(synthetic_data)} samples with {len([c for c in synthetic_data.columns if c.startswith('feature_')])} features")
print(synthetic_gen.get_dataset_info())


SyntheticDataGenerator creates ML-ready datasets:
  • Uses sklearn.make_classification for realistic features
  • Trains lightgbm model automatically
  • Generated 1000 samples with 10 features
{'n_samples': 2000, 'n_features': 10, 'n_informative': 6, 'fraud_rate': np.float64(0.019), 'n_fraud': np.int64(38), 'n_legitimate': np.int64(1962), 'class_balance': [0.985, 0.015]}


**LoggingPolicyGenerator**

Generates a logging policy (i.e. if a transaction would be blocked or not based on a model score, and with which probability) upon a dataset

In [37]:
policy_gen = LoggingPolicyGenerator(LoggingPolicyConfig(
    cutoff=0.05,
    exploration_rate=0.1,
    random_state=42
))

# Use the sample data from DataGenerator
policy_data = policy_gen.generate_policy(sample_data.head(100))  # Small sample for demo

print("LoggingPolicyGenerator simulates fraud prevention policies:")
print(f"  • Cutoff threshold: {policy_gen.config.cutoff} (block if score > threshold)")
print(f"  • Exploration rate: {policy_gen.config.exploration_rate} (% of blocked transactions to allow)")
print(f"  • Actions generated: {policy_data['model_action'].value_counts().to_dict()}")
print(f"  • Policy actions: {policy_data['policy_action'].value_counts().to_dict()}")
print(f"  • Propensity scores: [{policy_data['propensity_score'].min():.3f}, {policy_data['propensity_score'].max():.3f}]")

LoggingPolicyGenerator simulates fraud prevention policies:
  • Cutoff threshold: 0.05 (block if score > threshold)
  • Exploration rate: 0.1 (% of blocked transactions to allow)
  • Actions generated: {'allow': 80, 'block': 20}
  • Policy actions: {'allow': 82, 'block': 18}
  • Propensity scores: [0.000, 1.000]


**CounterfactualEstimator**

Calculates OPE metrics based on a logging policy, and a new policy (or the original one as well)

In [41]:
import numpy as np

estimator = CounterfactualEstimator(
    CounterfactualEstimatorConfig(n_bootstrap=100, random_state=42),  # Small bootstrap for demo
    policy_data
)

# policy_data[policy_data['policy_action'] == 'allow']
observed_data = estimator.observed_data

# Random binary decisions and prediction probabilities
random_policy = np.random.RandomState(42).randint(0, 2, len(observed_data))
random_policy_proba = np.random.RandomState(42).uniform(0, 1, len(observed_data))

# Estimate OPE metrics for this random policy
ope_results = estimator.estimate_ope_metrics(random_policy, random_policy_proba)

print(f"  • Random policy results:")
for metric_name, metric_data in ope_results.items():
    mean_val = metric_data.get('mean', 'N/A')
    ci_lower = metric_data.get('p025', 'N/A')
    ci_upper = metric_data.get('p975', 'N/A')
    print(f"    - {metric_name}: {mean_val:.4f} (95% CI: [{ci_lower:.4f}, {ci_upper:.4f}])")

  • Random policy results:
    - precision: 0.1900 (95% CI: [0.0000, 0.4670])
    - recall: 0.4912 (95% CI: [0.0000, 1.0000])
    - fraud_rate: 0.1950 (95% CI: [0.0000, 0.4862])
    - average_precision: 0.5313 (95% CI: [0.0435, 0.9580])
    - roc_auc: nan (95% CI: [nan, nan])
    - brier_score: 0.2711 (95% CI: [0.1996, 0.3318])


/Users/rbrosa/Documents/github_personal/counterfactual-fraud-model/.venv/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1201: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
/Users/rbrosa/Documents/github_personal/counterfactual-fraud-model/.venv/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1201: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(


**CounterfactualEstimator**

Wrappers over sklearn API to train models and calculate performance metrics

In [43]:
# Demonstrate concrete training using the synthetic data from above
print("\n  🎯 Example: Training models on synthetic data")
trainer = ModelTrainer()

# Extract features and target from the synthetic data generated above
feature_columns = [col for col in synthetic_data.columns if col.startswith('feature_')]
X = synthetic_data[feature_columns]
y = synthetic_data['is_fraud']

# Split into train/test sets for proper evaluation
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"  • Using {len(feature_columns)} features from synthetic dataset")
print(f"  • Train set: {len(X_train)} samples (fraud rate: {y_train.mean():.3f})")
print(f"  • Test set: {len(X_test)} samples (fraud rate: {y_test.mean():.3f})")

# Train and compare different model types
model_performance = {}
for model_type in [ModelType.LIGHTGBM, ModelType.RANDOM_FOREST, ModelType.LOGISTIC]:
    print(f"\n  📊 Training {model_type.value} model...")
    
    # Create model configuration
    model_config = ModelConfig(
        model_type=model_type,
        model_params={"random_state": 42, "n_estimators": 50} if model_type != ModelType.LOGISTIC else {"random_state": 42},
        random_state=42
    )
    
    # Train the model
    trained_model = trainer.train_model(X_train, y_train, model_config)
    
    # Evaluate on test set
    test_performance = trainer.calculate_performance(trained_model, X_test, y_test)
    model_performance[model_type.value] = test_performance
    
    print(f"    • ROC-AUC: {test_performance['roc_auc']:.4f}")
    print(f"    • Precision: {test_performance['precision']:.4f}")
    print(f"    • Recall: {test_performance['recall']:.4f}")
    print(f"    • F1-Score: {test_performance['f1']:.4f}")


  🎯 Example: Training models on synthetic data
  • Using 10 features from synthetic dataset
  • Train set: 700 samples (fraud rate: 0.019)
  • Test set: 300 samples (fraud rate: 0.020)

  📊 Training lightgbm model...
    • ROC-AUC: 0.8186
    • Precision: 0.0000
    • Recall: 0.0000
    • F1-Score: 0.0000

  📊 Training random_forest model...
    • ROC-AUC: 0.7463
    • Precision: 0.0000
    • Recall: 0.0000
    • F1-Score: 0.0000

  📊 Training logistic model...
    • ROC-AUC: 0.5935
    • Precision: 0.0000
    • Recall: 0.0000
    • F1-Score: 0.0000


## Pipeline Classes
Pipelines are the most important concrete implementaitons from the package. They use the configs shown above to generate synthetic data, train models, generate logging policies, calculate OPE metrics and so on.

Auxiliary functions

In [27]:
def print_result_summary(result, strategy_name=None):
    """Print a formatted summary of pipeline results."""
    prefix = f"[{strategy_name}] " if strategy_name else ""
    
    # Print key statistics
    if 'statistics' in result:
        stats = result['statistics']
        print(f"  {prefix}📈 Key Statistics:")
        print(f"    • Total transactions: {stats.get('total_transactions', 'N/A'):,}")
        print(f"    • Allow rate: {stats.get('allow_rate', 'N/A'):.3f}")
        print(f"    • Block rate: {stats.get('block_rate', 'N/A'):.3f}")
        print(f"    • Overall fraud rate: {stats.get('fraud_rate_overall', 'N/A'):.4f}")
        print(f"    • Fraud rate in allowed: {stats.get('fraud_rate_allowed', 'N/A'):.4f}")
    
    # Print OPE metrics
    if 'ope_metrics' in result:
        print(f"  {prefix}📊 Off-Policy Evaluation Metrics:")
        for metric_name, metric_data in result['ope_metrics'].items():
            mean_val = metric_data.get('mean', 'N/A')
            ci_lower = metric_data.get('p025', 'N/A')
            ci_upper = metric_data.get('p975', 'N/A')
            print(f"    • {metric_name}: {mean_val:.4f} (95% CI: [{ci_lower:.4f}, {ci_upper:.4f}])")

def print_model_performance(model_perf):
    """Print model performance metrics."""
    print("  🎯 Model Performance:")
    for metric, value in model_perf.items():
        print(f"    • {metric}: {value:.4f}")

**Off Policy Evaluation Pipeline**

This pipeline orchestrates the creation of model socres, fraud occurrence, and off policy estimation of model performance based on distributional parameters, logging policy parameters and bootstrap parameters

In [24]:
config = OffPolicyEvaluationConfig(
    data_generator=DataGeneratorConfig(
        sample_size=5_000,  # Small dataset for quick demo
        alpha=0.1,
        beta_param=2.0,
        random_state=42
    ),
    logging_policy=LoggingPolicyConfig(
        cutoff=0.05,
        exploration_rate=0.1,
        random_state=42
    ),
    counterfactual_estimator=CounterfactualEstimatorConfig(
        n_bootstrap=1000,  # Fewer iterations for speed
        random_state=42
    ),
    pipeline=PipelineConfig(include_data=False)
)

print("📊 Configuration:")
print(f"  • Sample size: {config.data_generator.sample_size:,}")
print(f"  • Cutoff threshold: {config.logging_policy.cutoff}")
print(f"  • Exploration rate: {config.logging_policy.exploration_rate}")
print(f"  • Bootstrap samples: {config.counterfactual_estimator.n_bootstrap:,}")

# Initialize and run pipeline
print("\n🚀 Running pipeline...")
pipeline = OffPolicyEvaluationPipeline(config)

# Run with default parameters (can also override here)
result = pipeline.run_pipeline(include_data=False)

print("\n✅ Results:")
print_result_summary(result)

📊 Configuration:
  • Sample size: 5,000
  • Cutoff threshold: 0.05
  • Exploration rate: 0.1
  • Bootstrap samples: 1,000

🚀 Running pipeline...

✅ Results:
  📈 Key Statistics:
    • Total transactions: 5,000
    • Allow rate: 0.831
    • Block rate: 0.169
    • Overall fraud rate: 0.0478
    • Fraud rate in allowed: 0.0397
  📊 Off-Policy Evaluation Metrics:
    • precision: 0.0901 (95% CI: [0.0361, 0.1566])
    • recall: 0.3321 (95% CI: [0.1666, 0.4779])
    • fraud_rate: 0.0386 (95% CI: [0.0327, 0.0448])
    • average_precision: 0.0888 (95% CI: [0.0416, 0.1608])
    • roc_auc: 0.5800 (95% CI: [0.4905, 0.6626])
    • brier_score: 0.0546 (95% CI: [0.0460, 0.0641])


**Synthetic Off Policy Evaluation Pipeline**

Similar to the above, but, instead of using distributional parameters, SyntheticDataConfig and ModelConfig configurations to manage synthetic data generation and model score behavior

In [28]:
# Create synthetic configuration
config = SyntheticOffPolicyEvaluationConfig(
    synthetic_data=SyntheticDataConfig(
        n_samples=10_000,     # Moderate size for demo
        n_features=15,
        n_informative=10,
        n_redundant=2,
        weights=[0.98, 0.02], # 2% fraud rate
        class_sep=1.0,
        random_state=42
    ),
    model=ModelConfig(
        model_type=ModelType.LIGHTGBM,
        model_params={
            "n_estimators": 50,  # Fewer trees for speed
            "max_depth": 4
        },
        random_state=42
    ),
    logging_policy=LoggingPolicyConfig(
        cutoff=0.05,
        exploration_rate=0.05,
        random_state=42
    ),
    counterfactual_estimator=CounterfactualEstimatorConfig(
        n_bootstrap=1000,
        random_state=42
    ),
    pipeline=PipelineConfig(include_data=False)
)

# Initialize and run pipeline
print("\n🚀 Running synthetic pipeline...")
pipeline = SyntheticOffPolicyEvaluationPipeline(config)

result = pipeline.run_pipeline(include_data=False)

print("\n✅ Results:")
print_result_summary(result)

# Show model performance if available
if 'model_performance' in result:
    print_model_performance(result['model_performance'])


🚀 Running synthetic pipeline...

✅ Results:
  📈 Key Statistics:
    • Total transactions: 5,000
    • Allow rate: 0.954
    • Block rate: 0.046
    • Overall fraud rate: 0.0244
    • Fraud rate in allowed: 0.0130
  📊 Off-Policy Evaluation Metrics:
    • precision: 0.2182 (95% CI: [0.0000, 0.4706])
    • recall: 0.4648 (95% CI: [0.0000, 0.7092])
    • fraud_rate: 0.0124 (95% CI: [0.0095, 0.0156])
    • average_precision: 0.4416 (95% CI: [0.0249, 0.7250])
    • roc_auc: 0.8360 (95% CI: [0.7124, 0.9123])
    • brier_score: 0.0169 (95% CI: [0.0111, 0.0254])
  🎯 Model Performance:
    • precision: 0.2603
    • recall: 0.5164
    • f1: 0.3462
    • roc_auc: 0.8525
    • average_precision: 0.4513


**Synthetic Retraining Pipeline**

Abstraction over SyntheticOffPolicyEvaluationPipeline. This pipeline wa buult to coordinate the retrainig of a model over a logging policy generate by SyntheticOffPolicyEvaluationPipeline by using different retraining strategies

In [29]:
# Create retraining configuration
base_config = SyntheticOffPolicyEvaluationConfig(
    synthetic_data=SyntheticDataConfig(
        n_samples=15_000,
        n_features=15,
        n_informative=10,
        n_redundant=2,  # Ensure sum is < n_features
        weights=[0.985, 0.015],
        random_state=42
    ),
    model=ModelConfig(model_type=ModelType.LIGHTGBM, random_state=42),
    logging_policy=LoggingPolicyConfig(
        cutoff=0.05,
        exploration_rate=0.1,
        random_state=42
    ),
    counterfactual_estimator=CounterfactualEstimatorConfig(
        n_bootstrap=1000,
        random_state=42
    )
)

config = SyntheticRetrainingConfig(
    base_config=base_config,
    retraining=RetrainingConfig(
        retrain_test_size=0.3,
        retrain_model=RetrainingModelConfig(
            base_model=ModelConfig(model_type=ModelType.LIGHTGBM, random_state=42),
            strategy=RetrainingStrategy.FILTERING,  # Will try different strategies
            classification_threshold=0.1
        )
    )
)

# Initialize pipeline
pipeline = SyntheticRetrainingPipeline(config)

# Generate the logging policy data first
print("\n🔄 Generating logging policy data...")
pipeline.generate_logging_policy_data()

# Try different retraining strategies
strategies_to_test = [RetrainingStrategy.FILTERING, RetrainingStrategy.WEIGHTING]

for strategy in strategies_to_test:
    print(f"\n🎯 Testing {strategy.value} strategy...")
    
    # Create retraining config for this strategy
    retraining_config = RetrainingConfig(
        retrain_test_size=0.3,
        retrain_model=RetrainingModelConfig(
            base_model=ModelConfig(model_type=ModelType.LIGHTGBM, random_state=42),
            strategy=strategy,
            classification_threshold=0.1
        )
    )
    
    result = pipeline.run_retrain_pipeline(retraining_config=retraining_config)
    
    print(f"\n✅ Results for {strategy.value}:")
    print_result_summary(result, strategy.value)


🔄 Generating logging policy data...

🎯 Testing filtering strategy...

✅ Results for filtering:
  [filtering] 📊 Off-Policy Evaluation Metrics:
    • precision: 0.0000 (95% CI: [0.0000, 0.0000])
    • recall: 0.0000 (95% CI: [0.0000, 0.0000])
    • fraud_rate: 0.0216 (95% CI: [0.0114, 0.0357])
    • average_precision: 0.0429 (95% CI: [0.0213, 0.0713])
    • roc_auc: 0.6244 (95% CI: [0.5435, 0.7170])
    • brier_score: 0.0215 (95% CI: [0.0114, 0.0356])

🎯 Testing weighting strategy...

✅ Results for weighting:
  [weighting] 📊 Off-Policy Evaluation Metrics:
    • precision: 0.5282 (95% CI: [0.0000, 1.0000])
    • recall: 0.1932 (95% CI: [0.0000, 0.5002])
    • fraud_rate: 0.0171 (95% CI: [0.0096, 0.0286])
    • average_precision: 0.1999 (95% CI: [0.0231, 0.5473])
    • roc_auc: 0.6613 (95% CI: [0.5008, 0.8304])
    • brier_score: 0.0206 (95% CI: [0.0115, 0.0331])
